In [2]:
# For data manipulation and visualization
import pandas as pd
import altair as alt

## Base assassino Shipman

In [3]:
shipman_victims = pd.read_csv(f'00-1-shipman-confirmed-victims-x.csv')

In [4]:
shipman_victims.head()

,DateofDeath,Name,Age,PlaceofDeath,Decision,yearOfDeath,gender,fractionalDeathYear,ageBracket,gender2
0,17-Mar-75,Eva Lyons,70,Own home,Unlawful killing,1975,0,1974.71,70-74,Women
1,07-Aug-78,Sarah Hannah Marsland,86,Own home,Unlawful killing,1978,0,1978.10,85-89,Women
2,30-Aug-78,Mary Ellen Jordan,73,Own home,Unlawful killing,1978,0,1978.16,70-74,Women
3,07-Dec-78,Harold Bramwell,73,Own home,Unlawful killing,1978,1,1978.44,70-74,Men
4,20-Dec-78,Annie Campbell,88,Own home,Unlawful killing,1978,0,1978.47,85-89,Women


### Replicando inteiramente o gráfico do caso, com diagrama de dispersão junto com histogramas

In [57]:
base = alt.Chart(shipman_victims)
base_bar = base.mark_bar()

xscale = alt.Scale(domain=(1974, 2000))
yscale = alt.Scale(domain=(40, 95))

# diagrama de dispersão de yearOfDeath e Age, com variável qualitativa (gender2) determinando as cores dos pontos
points = base.mark_circle().encode(
    alt.X("yearOfDeath",scale=xscale),
    alt.Y("Age",scale=yscale),
    color="gender2",
    tooltip=['yearOfDeath', 'Age', 'gender2']
)

# histograma da variável yearOfDeath
top_hist = (
    base_bar
    .encode(
        alt.X("yearOfDeath:Q", bin=alt.Bin(step=1, extent=xscale.domain), title=""),
        alt.Y("count()", stack=None, title=""),
        tooltip=['yearOfDeath', 'count()']
        #alt.Color("gender2:N"),
    )
    .properties(height=60)
)

# histograma da variável Age
right_hist = (
    base_bar
    .encode(
        alt.Y("Age:Q", bin=alt.Bin(step=5, extent=yscale.domain), title=""),
        alt.X("count()", stack=None, title=""),
        tooltip=['Age', 'count()']
        #alt.Color("gender2:N"),
    )
    .properties(width=60)
)

top_hist & (points | right_hist)

alt.VConcatChart(...)

### Tabela de dupla entrada

#### Formato matricial

In [11]:
tabela_dupla_entrada = shipman_victims.groupby(['gender2', 'ageBracket']).size().unstack(1)
tabela_dupla_entrada.loc['Total',:]= tabela_dupla_entrada.sum(axis=0)
tabela_dupla_entrada.loc[:,'Total'] = tabela_dupla_entrada.sum(axis=1)
tabela_dupla_entrada

ageBracket,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85-89,90-94,Total
gender2,,,,,,,,,,,,
Men,2.0,1.0,NaN,NaN,NaN,2.0,4.0,9.0,8.0,9.0,2.0,37.0
Women,NaN,2.0,6.0,5.0,6.0,16.0,34.0,38.0,44.0,22.0,5.0,178.0
Total,2.0,3.0,6.0,5.0,6.0,18.0,38.0,47.0,52.0,31.0,7.0,215.0


In [12]:
tabela_dupla_entrada.fillna(0, inplace=True)

ageBracket,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85-89,90-94,Total
gender2,,,,,,,,,,,,
Men,2.0,1.0,0.0,0.0,0.0,2.0,4.0,9.0,8.0,9.0,2.0,37.0
Women,0.0,2.0,6.0,5.0,6.0,16.0,34.0,38.0,44.0,22.0,5.0,178.0
Total,2.0,3.0,6.0,5.0,6.0,18.0,38.0,47.0,52.0,31.0,7.0,215.0


In [13]:
tabela_dupla_entrada

ageBracket,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85-89,90-94,Total
gender2,,,,,,,,,,,,
Men,2.0,1.0,0.0,0.0,0.0,2.0,4.0,9.0,8.0,9.0,2.0,37.0
Women,0.0,2.0,6.0,5.0,6.0,16.0,34.0,38.0,44.0,22.0,5.0,178.0
Total,2.0,3.0,6.0,5.0,6.0,18.0,38.0,47.0,52.0,31.0,7.0,215.0


#### Formato long

In [51]:
tabela_dupla_entrada_long = shipman_victims.groupby(['gender2', 'ageBracket']).size()
tabela_dupla_entrada_long

gender2  ageBracket
Men      40-44          2
         45-49          1
         65-69          2
         70-74          4
         75-79          9
         80-84          8
         85-89          9
         90-94          2
Women    45-49          2
         50-54          6
         55-59          5
         60-64          6
         65-69         16
         70-74         34
         75-79         38
         80-84         44
         85-89         22
         90-94          5
dtype: int64

#### Distribuição condicional de faixa etária a partir dos valores de gênero

In [52]:
(tabela_dupla_entrada_long / tabela_dupla_entrada_long.groupby(level=0).transform(sum) * 100)

gender2  ageBracket
Men      40-44          5.405405
         45-49          2.702703
         65-69          5.405405
         70-74         10.810811
         75-79         24.324324
         80-84         21.621622
         85-89         24.324324
         90-94          5.405405
Women    45-49          1.123596
         50-54          3.370787
         55-59          2.808989
         60-64          3.370787
         65-69          8.988764
         70-74         19.101124
         75-79         21.348315
         80-84         24.719101
         85-89         12.359551
         90-94          2.808989
dtype: float64

#### Distribuição condicional de gênero a partir dos valores de faixa etária

In [53]:
(tabela_dupla_entrada_long / tabela_dupla_entrada_long.groupby(level=1).transform(sum) * 100)

gender2  ageBracket
Men      40-44         100.000000
         45-49          33.333333
         65-69          11.111111
         70-74          10.526316
         75-79          19.148936
         80-84          15.384615
         85-89          29.032258
         90-94          28.571429
Women    45-49          66.666667
         50-54         100.000000
         55-59         100.000000
         60-64         100.000000
         65-69          88.888889
         70-74          89.473684
         75-79          80.851064
         80-84          84.615385
         85-89          70.967742
         90-94          71.428571
dtype: float64

### Gráficos de barras segmentadas

#### Da distribuição condicional de gênero a partir dos valores de faixa etária


##### Com contagem em valores absolutos

In [54]:
alt.Chart(tabela_dupla_entrada_long.reset_index().rename(columns={0:'contagem'})).mark_bar().encode(
    x=alt.X('contagem'),
    y='ageBracket',
    color='gender2'
)

alt.Chart(...)

##### Com percentuais

In [55]:
alt.Chart(tabela_dupla_entrada_long.reset_index().rename(columns={0:'contagem'})).mark_bar().encode(
    x=alt.X('contagem',stack="normalize"),
    y='ageBracket',
    color='gender2'
)

alt.Chart(...)

#### Da distribuição condicional de faixa etária a partir dos valores de gênero

In [56]:
alt.Chart(tabela_dupla_entrada_long.reset_index().rename(columns={0:'contagem'})).mark_bar().encode(
    x=alt.X('contagem',stack="normalize"),
    y='gender2',
    color='ageBracket'
)

alt.Chart(...)

### Extra: criando uma variável qualitativa a partir de uma variável quantitativa

In [7]:
shipman_victims['Age_Category'] = pd.cut(
    x=shipman_victims['Age'],
    bins=[0, 18, 30, 55, 100], # list(range(40, 95, 5)),
    labels=['Child', 'Young Adult', 'Adult', 'Senior'], # caso queira dar outros nomes para cada intervalo. se não colocar esse parâmetro, fica o intervalo.
    right=True  # intevalo fechado ou aberto na direita (valores do final do intervalo vão pertencer à categoria menor ou à maior?)
)

In [ ]:
shipman_victims.head()

## Base crianças caso Bristol

Hospitais com maior movimento têm taxas de sobrevivência maiores? - o "efeito de volume"

### Dados para crianças com idade inferior a um ano ao longo do período de 1991-5 - foco do inquérito público de Bristol

In [60]:
child_survive_91 = pd.read_csv(f'02-5-child-heart-surgery-1991-x.csv')

In [61]:
child_survive_91.head()

,Hospital,Operations,Survivors,Deaths,ThirtyDaySurvival,PercentageDying
0,Bristol,143,102,41,71.3,28.7
1,Leicester,187,162,25,86.6,13.4
2,Leeds,323,299,24,92.6,7.4
3,Oxford,122,99,23,81.1,18.9
4,Guys,164,139,25,84.8,15.2


#### Diagrama de dispersão de operações por taxa de sobrevivência

In [62]:
alt.Chart(child_survive_91).mark_circle().encode(
    alt.X("Operations"),
    alt.Y("ThirtyDaySurvival",scale=alt.Scale(domain=[60,100])),
    tooltip=['Hospital', 'Operations', 'ThirtyDaySurvival']
)

alt.Chart(...)

Claro ponto discrepante, um hospital menor com taxa de sobrevivência de apenas 71% - trata-se de Bristol.

Vamos calcular o coeficiente de Pearson com e sem esse ponto!

In [63]:
child_survive_91[['Operations','ThirtyDaySurvival']].corr(numeric_only=True)

,Operations,ThirtyDaySurvival
Operations,1.00000,0.58369
ThirtyDaySurvival,0.58369,1.00000


In [64]:
child_survive_91.drop([0])[['Operations','ThirtyDaySurvival']].corr(numeric_only=True) # tirando Bristol com drop([0]) - 0 é a linha da base que corresponde a Bristol

,Operations,ThirtyDaySurvival
Operations,1.000000,0.662775
ThirtyDaySurvival,0.662775,1.000000


Removendo Bristol ou não, o padrão dos dados sugere que há taxas de sobrevivência mais altas em hospitais que conduzem um número maior de operações.

### Dados para todas as crianças abaixo de 16 anos no período de 2012-5

In [65]:
child_survive_2012 = pd.read_csv(f'02-5-child-heart-surgery-2012-x.csv')

In [66]:
child_survive_2012.head()

,Hospital,Operations,Survivors,Deaths,ThirtyDaySurvival,PercentageDying
0,London - Harley Street,418,413,5,98.8,1.2
1,Leicester,607,593,14,97.7,2.3
2,Newcastle,668,653,15,97.8,2.2
3,Glasgow,760,733,27,96.4,3.6
4,Southampton,829,815,14,98.3,1.7


#### Diagrama de dispersão de operações por taxa de sobrevivência

In [67]:
alt.Chart(child_survive_2012).mark_circle().encode(
    alt.X("Operations"),
    alt.Y("ThirtyDaySurvival",scale=alt.Scale(domain=[95,100])),
    tooltip=['Hospital', 'Operations', 'ThirtyDaySurvival']
)

alt.Chart(...)

Aqui não parece mais ter qualquer relação clara entre o número de casos e as taxas de sobrevivência, e o coeficiente de Pearson mostra isso.

In [68]:
child_survive_2012[['Operations','ThirtyDaySurvival']].corr(numeric_only=True)

,Operations,ThirtyDaySurvival
Operations,1.000000,0.161525
ThirtyDaySurvival,0.161525,1.000000


No entanto, com tão poucos hospitais analisados, o coeficiente de correlação pode ser muito sensível a dados individuais - se removermos o menor hospital, que tem taxa de sobrevivência mais elevada, a correlação tem um salto considerável.

In [69]:
child_survive_2012.drop([0])[['Operations','ThirtyDaySurvival']].corr(numeric_only=True)

,Operations,ThirtyDaySurvival
Operations,1.000000,0.402118
ThirtyDaySurvival,0.402118,1.000000


## Base Datasaurus Dozen

Dados fictícios para os quais os coeficientes de correlação de Pearson são próximos de 0. Isso não significa que não haja nenhuma relação entre as duas variáveis representadas nos gráficos (exemplos de Alberto Cairo).

In [70]:
datasaurus_dozen = pd.read_csv(f'DatasaurusDozen.tsv', sep='\t')

In [71]:
datasaurus_dozen.head()

,dataset,x,y
0,dino,55.3846,97.1795
1,dino,51.5385,96.0256
2,dino,46.1538,94.4872
3,dino,42.8205,91.4103
4,dino,40.7692,88.3333


In [72]:
datasaurus_dozen.dataset.unique()

<ArrowStringArray>
[      'dino',       'away',    'h_lines',    'v_lines',    'x_shape',
       'star', 'high_lines',       'dots',     'circle',   'bullseye',
   'slant_up', 'slant_down', 'wide_lines']
Length: 13, dtype: str

In [73]:
dino = datasaurus_dozen[datasaurus_dozen.dataset=='dino']

In [74]:
alt.Chart(dino).mark_circle().encode(
    alt.X("x"),
    alt.Y("y")
)

alt.Chart(...)

In [75]:
dino.corr(numeric_only=True)

,x,y
x,1.000000,-0.064472
y,-0.064472,1.000000


In [76]:
slant_down = datasaurus_dozen[datasaurus_dozen.dataset=='slant_down']

In [77]:
alt.Chart(slant_down).mark_circle().encode(
    alt.X("x"),
    alt.Y("y")
)

alt.Chart(...)

In [78]:
dino.corr(numeric_only=True)

,x,y
x,1.000000,-0.064472
y,-0.064472,1.000000


In [79]:
alt.Chart(datasaurus_dozen).mark_circle().encode(
    x="x",
    y="y",
    column="dataset:N",
)

alt.Chart(...)

In [80]:
datasaurus_dozen.groupby('dataset').corr(numeric_only=True)

x         y
dataset                         
away       x  1.000000 -0.064128
           y -0.064128  1.000000
bullseye   x  1.000000 -0.068586
           y -0.068586  1.000000
circle     x  1.000000 -0.068343
           y -0.068343  1.000000
dino       x  1.000000 -0.064472
           y -0.064472  1.000000
dots       x  1.000000 -0.060341
           y -0.060341  1.000000
h_lines    x  1.000000 -0.061715
           y -0.061715  1.000000
high_lines x  1.000000 -0.068504
           y -0.068504  1.000000
slant_down x  1.000000 -0.068980
           y -0.068980  1.000000
slant_up   x  1.000000 -0.068609
           y -0.068609  1.000000
star       x  1.000000 -0.062961
           y -0.062961  1.000000
v_lines    x  1.000000 -0.069446
           y -0.069446  1.000000
wide_lines x  1.000000 -0.066575
           y -0.066575  1.000000
x_shape    x  1.000000 -0.065583
           y -0.065583  1.000000